# 🗳️ Self-Consistency: Sample Several Reasoning Paths and Vote

## Learning Objectives
In this notebook, you will learn:
1. **Why one sample is not enough** - A single Chain-of-Thought answer is one draw from a distribution of reasoning paths
2. **Sample and vote** - Ask the same question N times at temperature > 0, extract each final answer, take the majority
3. **Measuring the effect** - Compare single-sample accuracy with vote accuracy against ground truth computed in code
4. **Agreement as a confidence signal** - What the vote share tells you, and the errors voting cannot fix

## Where this fits
This is the first of three *sampling and search* notebooks. Self-consistency spends extra LLM calls on the **final answer only**: no tree, no evaluator, just independent samples and a vote. `02_tree_of_thoughts.ipynb` then spends them on **intermediate steps**, and `03_lats.ipynb` adds a search policy on top. Module 07's voting (`modules/07_parallelization/notebooks/02_parallelization_sectioning_and_voting.ipynb`) is a different idea: there, *differently prompted* reviewers vote; here, one identical prompt is sampled several times.

## Prerequisites
- Chain-of-Thought prompting and structured output - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Wang et al. (Google), *Self-Consistency Improves Chain of Thought Reasoning in Language Models* (2022). [arXiv:2203.11171](https://arxiv.org/abs/2203.11171) |

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: One model sampled at temperature 0.7
# ============================================================================
from aied_helpers import get_openai_llm

# temperature > 0 is the whole point: at 0 every sample would follow the same path
sampling_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {sampling_llm.model_name} (temperature={sampling_llm.temperature})")

---
## 📖 Part 1: Why One Sample Is Not Enough

A single call to an LLM with Chain-of-Thought prompting is a **single sample from a distribution over possible reasoning paths**. At temperature 0 that distribution collapses to one path, which may or may not be correct. At a non-zero temperature, re-asking the same question produces *different* reasoning paths — some correct, some flawed in different ways.

Self-Consistency exploits this: instead of trusting one sampled path, it draws **N independent samples**, lets each one reason through the problem on its own (no path sees any other path), extracts each sample's final answer, and then takes a **majority vote**. When the mistakes are *random*, they rarely agree with each other while correct paths converge on the same answer, so the vote washes out noise.

### High-level Workflow
1. **Fix the question, vary the sampling.** The same CoT prompt is sent N times with `temperature > 0`.
2. **Independent reasoning.** No sample sees another sample's reasoning — unlike a debate or a reflection loop.
3. **Extract the final answer** from each free-form trace (here: a `Final Answer: <number>` line).
4. **Vote.** Count how often each distinct answer occurs and return the most common one.

### Strengths & Weaknesses
* **Strengths:** a loop and a `Counter` — no extra model, no verifier, no training; the N calls are independent, so they can run concurrently.
* **Weaknesses:** N times the cost; answers must be short and comparable (numbers, labels — not essays); and it corrects **random** errors, not **systematic** ones (Part 4 shows one).

---
## ✍️ Part 2: The Prompt and the Answer Extractor

The prompt asks for step-by-step reasoning and one final line in a fixed format, so the vote can compare answers exactly. The extractor falls back to the last number in the text if the model forgets the format.

In [ ]:
# ============================================================================
# COT PROMPT + FINAL-ANSWER EXTRACTOR
# ============================================================================
import re

COT_PROMPT = (
    "Solve the following problem. Think step by step, showing your arithmetic explicitly. "
    "On the very last line, output ONLY the line:\n"
    "Final Answer: <number>\n\n"
    "Problem: {question}"
)

FINAL_ANSWER_RE = re.compile(r"Final Answer:\s*\**\s*([\-0-9.,]+)", re.IGNORECASE)


def extract_final_answer(text: str) -> str:
    """Pull the value after 'Final Answer:'; fall back to the last number in the text."""
    matches = FINAL_ANSWER_RE.findall(text)
    if matches:
        return matches[-1].strip().rstrip(".").replace(",", "")
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)
    return numbers[-1] if numbers else "UNPARSEABLE"


print(extract_final_answer("... so 47 - 15 - 30 = 2.\nFinal Answer: 2"))
print(extract_final_answer("The answer is 1,024"))

---
## 🗳️ Part 3: Sample N Times and Vote

`self_consistency()` sends the same prompt N times with `batch()` (the calls run concurrently), extracts every answer and tallies them with `collections.Counter`. It also returns the **agreement** — the share of samples that voted for the winner — which Part 4 uses as a confidence signal.

We start with a word problem whose answer we can compute in code.

In [ ]:
# ============================================================================
# SELF-CONSISTENCY: N independent samples + majority vote
# ============================================================================
from collections import Counter


def self_consistency(question: str, n: int = 5) -> dict:
    responses = sampling_llm.batch([COT_PROMPT.format(question=question)] * n)
    answers = [extract_final_answer(r.content) for r in responses]
    votes = Counter(answers)
    majority, count = votes.most_common(1)[0]
    return {"answers": answers, "votes": votes, "majority": majority,
            "agreement": count / n, "texts": [r.content for r in responses]}


BAKERY = (
    "A bakery starts the day with 23 cupcakes on the shelf. Partway through the morning they bake "
    "4 more trays, each holding 6 cupcakes. They sell 15 cupcakes in the morning, and in the afternoon "
    "they sell twice as many cupcakes as they sold in the morning. How many cupcakes are left at the end of the day?"
)
BAKERY_TRUTH = str(23 + 4 * 6 - 15 - 2 * 15)  # ground truth computed in code

result = self_consistency(BAKERY, n=5)
print("--- One sampled reasoning path ---")
print(result["texts"][0])
print("\n--- Vote tally ---")
for answer, count in result["votes"].most_common():
    print(f"  {answer!r}: {count} vote(s)")
print(f"\nMajority: {result['majority']} | agreement {result['agreement']:.0%} | ground truth {BAKERY_TRUTH}")

gpt-4o-mini solves this one on (almost) every sample, so the vote is unanimous and adds nothing but cost. That is the first lesson: **self-consistency only pays off where single samples disagree**. The next part looks for questions where they do.

---
## 📏 Part 4: Measuring It Against Ground Truth

Three questions, each with an answer computed in code:

| Question | What we saw when prototyping (20 samples each) |
|---|---|
| Bakery word problem | correct every time |
| Sum of all digits in 1–100 | correct ~80% of the time; the wrong answers were scattered (**random** errors) |
| Three-digit numbers whose digits add up to 10 | correct ~40%; most wrong answers were the *same* number, 55 (**systematic** error) |

For each question we run **3 trials of N = 5 samples**: single-sample accuracy is measured over all 15 samples, vote accuracy over the 3 majority votes. Your numbers will vary from run to run — read them from the table, not from this text.

In [ ]:
# ============================================================================
# EXPERIMENT: single-sample accuracy vs majority-vote accuracy
# ============================================================================
QUESTIONS = {
    "bakery": (BAKERY, BAKERY_TRUTH),
    "digit_sum_1_to_100": (
        "What is the sum of all the digits in all the integers from 1 to 100?",
        str(sum(sum(map(int, str(i))) for i in range(1, 101))),
    ),
    "three_digit_sum_10": (
        "How many three-digit numbers have digits that add up to 10?",
        str(sum(1 for i in range(100, 1000) if sum(map(int, str(i))) == 10)),
    ),
}
N_SAMPLES, N_TRIALS = 5, 3

rows = []
for name, (question, truth) in QUESTIONS.items():
    trials = [self_consistency(question, n=N_SAMPLES) for _ in range(N_TRIALS)]
    all_answers = [a for t in trials for a in t["answers"]]
    rows.append({
        "question": name,
        "truth": truth,
        "single": sum(a == truth for a in all_answers),
        "votes_ok": sum(t["majority"] == truth for t in trials),
        "majorities": [t["majority"] for t in trials],
        "agreement": sum(t["agreement"] for t in trials) / N_TRIALS,
        "answers": Counter(all_answers).most_common(4),
    })

print(f"{'question':<20}{'truth':>6}{'single-sample':>16}{'vote':>8}{'mean agreement':>17}   majorities")
for r in rows:
    print(f"{r['question']:<20}{r['truth']:>6}{r['single']:>9}/{N_SAMPLES * N_TRIALS:<6}"
          f"{r['votes_ok']:>5}/{N_TRIALS}{r['agreement']:>15.0%}     {r['majorities']}")
print()
for r in rows:
    print(f"{r['question']:<20} most common answers: {r['answers']}")

Sampling is random, so a question can behave differently from one run to the next. Instead of hard-coding the story, the next cell reads each row and says which of the three cases it shows:
- **Easy** — every sample is right: the extra samples were wasted.
- **Random errors** — the wrong answers disagree with each other, so the vote beats a single sample. This is the case self-consistency is designed for.
- **Systematic error** — the wrong answers agree, so the vote returns a wrong number *more* reliably than a single sample would.

In [ ]:
# ============================================================================
# INTERPRETATION: which case does each row show in this run?
# ============================================================================
for r in rows:
    single_acc = r["single"] / (N_SAMPLES * N_TRIALS)
    vote_acc = r["votes_ok"] / N_TRIALS
    wrong = [(a, n) for a, n in r["answers"] if a != r["truth"]]
    top_wrong = wrong[0][1] if wrong else 0
    if single_acc == 1:
        case = "easy: every sample right, voting only added cost"
    elif vote_acc > single_acc:
        case = f"random errors: vote {vote_acc:.0%} beats single sample {single_acc:.0%}"
    elif top_wrong > r["single"]:
        case = f"systematic error: the wrong answer {wrong[0][0]} got more votes than the truth"
    else:
        case = f"no gain: vote {vote_acc:.0%} vs single sample {single_acc:.0%}"
    print(f"{r['question']:<20} -> {case}")

---
## 📶 Part 5: Agreement as a Confidence Signal

The `agreement` column is a free by-product: when the samples agree unanimously the answer is usually either right (bakery) or *systematically* wrong, and when agreement is low the model is guessing. A practical use is to **act on high agreement and escalate low agreement** — to a stronger model, a tool (here: the few lines of Python that computed our ground truth), or a human. `05_reflexive_metacognitive.ipynb` builds that kind of "should I answer this myself?" decision into an agent.

In [ ]:
# ============================================================================
# AGREEMENT THRESHOLD: answer or escalate
# ============================================================================
AGREEMENT_THRESHOLD = 0.8

for r in rows:
    decision = "answer" if r["agreement"] >= AGREEMENT_THRESHOLD else "escalate (low agreement)"
    print(f"{r['question']:<20} agreement {r['agreement']:.0%} -> {decision}")

Note what the threshold can and cannot do: a low-agreement question is flagged, but a question the model gets wrong *consistently* can still pass the threshold. Agreement measures the model's consistency, not the truth — only ground truth (or a tool) can tell those apart.

---
## 📝 Summary

In this notebook, we learned:

### 1. Why one sample is not enough
- A CoT answer is one draw from a distribution of reasoning paths; at temperature > 0 re-asking gives different paths.

### 2. Sample and vote
- Same prompt, N concurrent samples, extract a comparable final answer, `Counter.most_common(1)`.

### 3. Measuring the effect
- Measured against ground truth computed in code: voting helps when errors are random, adds only cost when the model is already right, and **locks in** a systematic error.

### 4. Agreement as a confidence signal
- The vote share is a cheap uncertainty estimate: act on high agreement, escalate low agreement — but consistent is not the same as correct.

### Next Steps
- **`02_tree_of_thoughts.ipynb`**: instead of voting on final answers, branch and evaluate *intermediate* steps, and search the resulting tree